# Player Similarity Scouting — data pipeline

Builds the dataset used by the Streamlit app: pulls raw event data from StatsBomb Open Data (World Cup 2022 + Euro 2024), recomputes xG, progressive passes/carries and defensive stats from the raw coordinates, clusters players by playing style, and saves everything to `data/player_profiles.pkl`.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from statsbombpy import sb

pd.set_option("display.max_columns", 50)

## 2. Competitions and matches

Free StatsBomb Open Data access only covers a handful of competitions (no full domestic seasons for the big leagues). Went with the 2022 World Cup and Euro 2024 since international tournaments still give a decent spread of players across the top leagues.

In [ ]:
competitions = sb.competitions()

wc_row = competitions[
    (competitions["competition_name"] == "FIFA World Cup") & (competitions["season_name"] == "2022")
].iloc[0]
euro_row = competitions[
    (competitions["competition_name"] == "UEFA Euro") & (competitions["season_name"] == "2024")
].iloc[0]

matches_wc = sb.matches(competition_id=wc_row["competition_id"], season_id=wc_row["season_id"])
matches_euro = sb.matches(competition_id=euro_row["competition_id"], season_id=euro_row["season_id"])

all_matches = pd.concat([
    matches_wc.assign(competition="WC2022"),
    matches_euro.assign(competition="Euro2024"),
])

all_matches.shape

## 3. Pulling all events

The slow part, 115 matches of event data. Cached to parquet so reruns don't hit the API again.

In [ ]:
# quick look at one match before running the full loop, just to check the columns
# sample_match_id = all_matches.iloc[0]["match_id"]
# test_events = sb.events(match_id=sample_match_id)
# test_events["type"].value_counts()

events_path = Path("data/events_wc2022_euro2024.parquet")

if events_path.exists():
    all_events = pd.read_parquet(events_path)
else:
    events_list = []
    for i, match_id in enumerate(all_matches["match_id"]):
        events = sb.events(match_id=match_id)
        events["match_id"] = match_id
        events_list.append(events)
        if (i + 1) % 10 == 0:
            print(f"{i + 1} / {len(all_matches)} matches done")

    all_events = pd.concat(events_list, ignore_index=True)
    events_path.parent.mkdir(exist_ok=True)
    all_events.to_parquet(events_path)

all_events.shape

## 4. Minutes played

Not given directly by StatsBomb, so it has to be rebuilt from `Starting XI` and `Substitution` events.

In [ ]:
def get_match_minutes(events, match_id):
    match_events = events[events["match_id"] == match_id]
    duration = match_events["minute"].max()

    starting = match_events[match_events["type"] == "Starting XI"]
    minutes = {}

    for _, row in starting.iterrows():
        for player in row["tactics"]["lineup"]:
            minutes[player["player"]["id"]] = {"name": player["player"]["name"], "on": 0, "off": duration}

    subs = match_events[match_events["type"] == "Substitution"]
    for _, row in subs.iterrows():
        off_id = row["player_id"]
        sub_minute = row["minute"]
        if off_id in minutes:
            minutes[off_id]["off"] = sub_minute
        if pd.notna(row.get("substitution_replacement_id")):
            on_id = row["substitution_replacement_id"]
            minutes[on_id] = {"name": row["substitution_replacement"], "on": sub_minute, "off": duration}

    return pd.DataFrame([
        {"match_id": match_id, "player_id": pid, "player_name": v["name"], "minutes_played": v["off"] - v["on"]}
        for pid, v in minutes.items()
    ])

minutes_list = [get_match_minutes(all_events, mid) for mid in all_matches["match_id"]]
player_minutes = pd.concat(minutes_list, ignore_index=True)

player_minutes_total = (
    player_minutes.groupby(["player_id", "player_name"])["minutes_played"]
    .sum()
    .reset_index()
)

player_minutes_total.sort_values("minutes_played", ascending=False).head()

## 5. Features recomputed from the raw events

This is the main point of using StatsBomb instead of a pre-aggregated source: xG is already included per shot, but progressive passes/carries and the defensive numbers are all built directly from each event's start and end coordinates.

In [ ]:
def dist_to_goal(loc):
    if loc is None or (isinstance(loc, float) and np.isnan(loc)):
        return np.nan
    x, y = loc
    return np.sqrt((120 - x) ** 2 + (40 - y) ** 2)

def is_valid_loc(loc):
    # events reloaded from parquet come back as numpy arrays instead of plain python lists,
    # so an isinstance(loc, list) check here quietly breaks after a reload -> use this instead
    if loc is None:
        return False
    if isinstance(loc, float) and np.isnan(loc):
        return False
    return hasattr(loc, "__len__") and len(loc) == 2

def is_progressive(start, end):
    d_start = dist_to_goal(start)
    d_end = dist_to_goal(end)
    if np.isnan(d_start) or np.isnan(d_end):
        return False
    gain = d_start - d_end
    own_half = start[0] < 60
    opp_half = end[0] >= 60
    if own_half and not opp_half:
        return gain >= 30
    elif own_half and opp_half:
        return gain >= 15
    else:
        return gain >= 10

ev = all_events.copy()

shots = ev[ev["type"] == "Shot"].copy()
shots["is_goal"] = shots["shot_outcome"] == "Goal"
shot_stats = shots.groupby("player_id").agg(
    shots=("id", "count"),
    goals=("is_goal", "sum"),
    xg=("shot_statsbomb_xg", "sum"),
).reset_index()

passes = ev[ev["type"] == "Pass"].copy()
passes["is_progressive"] = passes.apply(
    lambda r: is_progressive(r["location"], r["pass_end_location"])
    if is_valid_loc(r["location"]) and is_valid_loc(r["pass_end_location"]) else False,
    axis=1,
)
pass_stats = passes.groupby("player_id").agg(
    passes_completed=("pass_outcome", lambda x: x.isna().sum()),
    progressive_passes=("is_progressive", "sum"),
).reset_index()

carries = ev[ev["type"] == "Carry"].copy()
carries["is_progressive"] = carries.apply(
    lambda r: is_progressive(r["location"], r["carry_end_location"])
    if is_valid_loc(r["location"]) and is_valid_loc(r["carry_end_location"]) else False,
    axis=1,
)
carry_stats = carries.groupby("player_id").agg(progressive_carries=("is_progressive", "sum")).reset_index()

defensive_types = ["Interception", "Clearance", "Block", "Ball Recovery", "Pressure"]
defense = ev[ev["type"].isin(defensive_types)]
defense_stats = defense.groupby(["player_id", "type"]).size().unstack(fill_value=0).reset_index()

player_profiles = (
    player_minutes_total
    .merge(shot_stats, on="player_id", how="left")
    .merge(pass_stats, on="player_id", how="left")
    .merge(carry_stats, on="player_id", how="left")
    .merge(defense_stats, on="player_id", how="left")
    .fillna(0)
)

player_profiles.head()

## 6. Discipline: cards and fouls

Cards can come from two different event types in StatsBomb: one tied to a `Foul Committed`, or a standalone `Bad Behaviour` event (e.g. a second yellow for dissent, not linked to a specific foul). Missed the second one at first and undercounted red cards as a result.

In [ ]:
bad_behaviour = ev[ev["type"] == "Bad Behaviour"].copy()
bad_behaviour["is_card"] = bad_behaviour["bad_behaviour_card"].notna()
bad_behaviour["is_red"] = bad_behaviour["bad_behaviour_card"].isin(["Red Card", "Second Yellow"])

fouls = ev[ev["type"] == "Foul Committed"].copy()
fouls["is_card"] = fouls["foul_committed_card"].notna()
fouls["is_red"] = fouls["foul_committed_card"].isin(["Red Card", "Second Yellow"])

cards_from_fouls = fouls.groupby("player_id").agg(
    fouls_committed=("id", "count"),
    cards_from_fouls=("is_card", "sum"),
    red_cards=("is_red", "sum"),
).reset_index()

cards_from_bad_behaviour = bad_behaviour.groupby("player_id").agg(
    cards_from_bad_behaviour=("is_card", "sum"),
    red_from_bad_behaviour=("is_red", "sum"),
).reset_index()

fouls_won_stats = ev[ev["type"] == "Foul Won"].groupby("player_id").size().reset_index(name="fouls_won")

player_profiles = (
    player_profiles
    .merge(cards_from_fouls, on="player_id", how="left")
    .merge(cards_from_bad_behaviour, on="player_id", how="left")
    .merge(fouls_won_stats, on="player_id", how="left")
    .fillna(0)
)

player_profiles["total_cards"] = player_profiles["cards_from_fouls"] + player_profiles["cards_from_bad_behaviour"]
player_profiles["red_cards"] = player_profiles["red_cards"] + player_profiles["red_from_bad_behaviour"]
player_profiles["yellow_cards"] = player_profiles["total_cards"] - player_profiles["red_cards"]

# Dumfries got a second yellow vs Argentina in the WC -> good sanity check that this is counted right
player_profiles[player_profiles["player_name"].str.contains("Dumfries")][["player_name", "total_cards", "yellow_cards", "red_cards"]]

## 7. Minutes filter and per-90 normalization

Dropping anyone under 450 minutes across both tournaments, so a few cameo appearances don't produce misleading per-90 numbers.

In [ ]:
MIN_MINUTES = 450

df = player_profiles[player_profiles["minutes_played"] >= MIN_MINUTES].copy()
df["nineties"] = df["minutes_played"] / 90

per90_source_cols = ["shots", "goals", "xg", "passes_completed", "progressive_passes",
                      "progressive_carries", "Ball Recovery", "Block", "Clearance", "Interception",
                      "Pressure", "fouls_committed"]

for col in per90_source_cols:
    df[f"{col}_p90"] = df[col] / df["nineties"]

df["cards_p90"] = df["total_cards"] / df["nineties"]

df.shape

## 8. Positions, teams and competition

In [ ]:
lineups_list = []
for match_id in all_matches["match_id"]:
    lineup = sb.lineups(match_id=match_id)
    for team_name, team_df in lineup.items():
        team_df = team_df.copy()
        team_df["team"] = team_name
        lineups_list.append(team_df[["player_id", "player_name", "positions", "team"]])

all_lineups = pd.concat(lineups_list, ignore_index=True)

def main_position(positions):
    if not positions:
        return None
    return positions[0]["position"]

all_lineups["position"] = all_lineups["positions"].apply(main_position)
all_lineups_valid = all_lineups.dropna(subset=["position"])

def most_common_position(x):
    counts = x.value_counts()
    return counts.idxmax() if len(counts) > 0 else None

player_positions = (
    all_lineups_valid.groupby("player_id")["position"]
    .agg(most_common_position)
    .reset_index()
)

all_teams = all_lineups.drop_duplicates(subset="player_id")[["player_id", "team"]]

df = df.merge(player_positions, on="player_id", how="left")
df = df.merge(all_teams, on="player_id", how="left")

def position_group(pos):
    if pos is None or pd.isna(pos):
        return "Unknown"
    pos = str(pos)
    if "Goalkeeper" in pos:
        return "Goalkeeper"
    if "Back" in pos:
        return "Defender"
    if "Midfield" in pos:
        return "Midfielder"
    if "Wing" in pos or "Forward" in pos:
        return "Attacker"
    return "Other"

df["position_group"] = df["position"].apply(position_group)

competition_map = all_matches.set_index("match_id")["competition"].to_dict()
player_competition = (
    player_minutes[["match_id", "player_id"]]
    .assign(competition=lambda x: x["match_id"].map(competition_map))
    .groupby("player_id")["competition"]
    .agg(lambda x: " + ".join(sorted(set(x))))
    .reset_index()
)
df = df.merge(player_competition, on="player_id", how="left")

df[["player_name", "position", "position_group", "team", "competition"]].head()

## 9. Scaling, PCA and clustering

No position constraint on the clustering on purpose, letting the model find playing-style groups on its own was the whole point (modern midfielders especially tend to blur position lines).

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

radar_cols = ["shots_p90", "goals_p90", "xg_p90", "passes_completed_p90", "progressive_passes_p90",
              "progressive_carries_p90", "Ball Recovery_p90", "Interception_p90", "Pressure_p90"]

X = df[radar_cols].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=0.90)
X_pca = pca.fit_transform(X_scaled)
print("components kept:", X_pca.shape[1])

for k in range(3, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca)
    print(k, round(silhouette_score(X_pca, labels), 3))

k=4 gave almost the same silhouette score as k=3, but with enough room to actually separate different attacking/defensive styles instead of lumping everything together.

In [ ]:
k_final = 4
km_final = KMeans(n_clusters=k_final, random_state=42, n_init=10)
df["cluster"] = km_final.fit_predict(X_pca)

df["cluster"].value_counts().sort_index()

In [ ]:
# 2D projection, just for the style map in the app
pca_2d = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
df["pca_x"] = pca_2d[:, 0]
df["pca_y"] = pca_2d[:, 1]

## 10. Similarity matrix

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(X_pca)
similarity_matrix.shape

## 11. Percentiles by position group

In [ ]:
percentile_cols = [c for c in df.columns if c.endswith("_p90")]

for col in percentile_cols:
    df[f"{col}_pct"] = df.groupby("position_group")[col].rank(pct=True) * 100

df[["player_name", "position_group", "xg_p90_pct"]].head()

## 12. Player age via Wikidata

No age or market value in StatsBomb. Transfermarkt would be the natural source for market value but it blocks scraping, so that's left out entirely (noted as a limitation in the app). Age is pulled from Wikidata instead, which is free and doesn't have the same anti-bot issues FBref ran into.

In [ ]:
import requests
import time

WIKI_HEADERS = {"User-Agent": "DataScienceFootProject/1.0 (student project, Polytech Nice)"}

def get_birthdate(player_name):
    try:
        search = requests.get(
            "https://www.wikidata.org/w/api.php",
            params={"action": "wbsearchentities", "search": player_name, "language": "en", "format": "json", "type": "item"},
            headers=WIKI_HEADERS, timeout=5,
        ).json()
        if not search.get("search"):
            return None
        entity_id = search["search"][0]["id"]

        claims = requests.get(
            "https://www.wikidata.org/w/api.php",
            params={"action": "wbgetclaims", "entity": entity_id, "property": "P569", "format": "json"},
            headers=WIKI_HEADERS, timeout=5,
        ).json()
        dob_str = claims["claims"]["P569"][0]["mainsnak"]["datavalue"]["value"]["time"]
        return dob_str[1:11]
    except Exception:
        return None

birthdates = {}
for name in df["player_name"]:
    birthdates[name] = get_birthdate(name)
    time.sleep(0.3)  # staying polite with the API rather than hammering it

df["birthdate"] = pd.to_datetime(df["player_name"].map(birthdates), errors="coerce")
today = pd.Timestamp.today()
df["age"] = ((today - df["birthdate"]).dt.days / 365.25).round(1)

df[["player_name", "age"]].head()

## 13. Save

In [ ]:
import pickle

with open("data/player_profiles.pkl", "wb") as f:
    pickle.dump({
        "df": df,
        "similarity_matrix": similarity_matrix,
        "radar_cols": radar_cols,
    }, f)

print("saved:", df.shape)